# 💳 Credit Card Fraud Detection
### Machine Learning Project
**Goal:** Detect fraudulent credit card transactions using Machine Learning

**Tech Stack:** Python, Pandas, Scikit-learn, Matplotlib, Seaborn

**Models Used:** Logistic Regression, Random Forest, XGBoost

## Step 1: Install & Import Libraries

In [ ]:
# Install required libraries
!pip install xgboost imbalanced-learn

In [ ]:
# Import all libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

## Step 2: Load Dataset
> We use a simulated dataset that mimics real credit card fraud data

In [ ]:
# Generate realistic credit card fraud dataset
np.random.seed(42)
n_samples = 10000
n_fraud = 200  # Only 2% fraud - real world scenario

# Legitimate transactions
legit = pd.DataFrame({
    'transaction_amount': np.random.normal(500, 200, n_samples - n_fraud).clip(10, 5000),
    'time_of_day': np.random.randint(0, 24, n_samples - n_fraud),
    'distance_from_home': np.random.normal(10, 5, n_samples - n_fraud).clip(0, 100),
    'num_transactions_today': np.random.randint(1, 8, n_samples - n_fraud),
    'is_foreign_transaction': np.random.choice([0, 1], n_samples - n_fraud, p=[0.95, 0.05]),
    'is_online': np.random.choice([0, 1], n_samples - n_fraud, p=[0.6, 0.4]),
    'credit_score': np.random.normal(700, 50, n_samples - n_fraud).clip(300, 850),
    'age': np.random.randint(22, 70, n_samples - n_fraud),
    'is_fraud': 0
})

# Fraudulent transactions (different patterns)
fraud = pd.DataFrame({
    'transaction_amount': np.random.normal(2000, 800, n_fraud).clip(100, 8000),
    'time_of_day': np.random.choice([1, 2, 3, 4, 23], n_fraud),  # Late night
    'distance_from_home': np.random.normal(80, 20, n_fraud).clip(10, 200),
    'num_transactions_today': np.random.randint(5, 20, n_fraud),
    'is_foreign_transaction': np.random.choice([0, 1], n_fraud, p=[0.3, 0.7]),
    'is_online': np.random.choice([0, 1], n_fraud, p=[0.2, 0.8]),
    'credit_score': np.random.normal(580, 60, n_fraud).clip(300, 750),
    'age': np.random.randint(18, 45, n_fraud),
    'is_fraud': 1
})

# Combine datasets
df = pd.concat([legit, fraud], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'✅ Dataset created with {len(df)} transactions')
print(f'📊 Fraud cases: {df["is_fraud"].sum()} ({df["is_fraud"].mean()*100:.1f}%)')
df.head(10)

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=== Dataset Info ===')
print(f'Shape: {df.shape}')
print(f'\nMissing Values:\n{df.isnull().sum()}')
print(f'\nBasic Statistics:')
df.describe()

In [ ]:
# Visualize class imbalance
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Class distribution
fraud_counts = df['is_fraud'].value_counts()
axes[0].pie(fraud_counts, labels=['Legitimate', 'Fraud'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0].set_title('Transaction Distribution', fontsize=14, fontweight='bold')

# Plot 2: Transaction Amount by fraud
df.boxplot(column='transaction_amount', by='is_fraud', ax=axes[1],
           boxprops=dict(color='#3498db'))
axes[1].set_title('Transaction Amount: Fraud vs Legit', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Is Fraud (0=No, 1=Yes)')
axes[1].set_ylabel('Amount (₹)')

# Plot 3: Foreign transactions
fraud_foreign = df.groupby(['is_fraud', 'is_foreign_transaction']).size().unstack()
fraud_foreign.plot(kind='bar', ax=axes[2], color=['#3498db', '#e74c3c'])
axes[2].set_title('Foreign Transactions vs Fraud', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Is Fraud')
axes[2].set_xticklabels(['Legitimate', 'Fraud'], rotation=0)
axes[2].legend(['Domestic', 'Foreign'])

plt.suptitle('Credit Card Fraud Detection - EDA', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved!')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Correlation heatmap saved!')

## Step 4: Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

# Train-Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Handle Class Imbalance using SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f'✅ Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')
print(f'✅ After SMOTE - Fraud: {y_train_resampled.sum()} | Legit: {(y_train_resampled==0).sum()}')

## Step 5: Model Training

In [ ]:
# Train 3 Models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {'model': model, 'y_pred': y_pred, 'y_prob': y_prob, 'auc': auc}
    print(f'✅ {name} trained | AUC-ROC: {auc:.4f}')

## Step 6: Model Evaluation

In [ ]:
# Detailed evaluation for each model
for name, result in results.items():
    print(f'\n=== {name} ===')
    print(classification_report(y_test, result['y_pred'],
                                target_names=['Legitimate', 'Fraud']))

In [ ]:
# Confusion Matrix for best model (XGBoost)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, result) in enumerate(results.items()):
    cm = confusion_matrix(y_test, result['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Legitimate', 'Fraud'],
                yticklabels=['Legitimate', 'Fraud'])
    axes[idx].set_title(f'{name}\nAUC: {result["auc"]:.4f}', fontweight='bold')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices - All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrices saved!')

In [ ]:
# ROC Curve Comparison
plt.figure(figsize=(8, 6))
colors = ['#3498db', '#2ecc71', '#e74c3c']

for (name, result), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, result['y_prob'])
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'{name} (AUC = {result["auc"]:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ ROC curves saved!')

In [ ]:
# Feature Importance (Random Forest)
rf_model = results['Random Forest']['model']
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(feature_importance['feature'], feature_importance['importance'],
         color='#3498db', edgecolor='white')
plt.xlabel('Feature Importance Score', fontsize=12)
plt.title('Feature Importance - Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance plot saved!')

## Step 7: Model Comparison Summary

In [ ]:
# Final comparison table
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

summary = []
for name, result in results.items():
    summary.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, result['y_pred']),
        'Precision': precision_score(y_test, result['y_pred']),
        'Recall': recall_score(y_test, result['y_pred']),
        'F1 Score': f1_score(y_test, result['y_pred']),
        'AUC-ROC': result['auc']
    })

summary_df = pd.DataFrame(summary).set_index('Model')
summary_df = summary_df.round(4)
print('=== 🏆 Model Comparison Summary ===')
print(summary_df.to_string())

best_model = summary_df['AUC-ROC'].idxmax()
print(f'\n🏆 Best Model: {best_model} with AUC-ROC of {summary_df.loc[best_model, "AUC-ROC"]:.4f}')

## Step 8: Predict on New Transaction

In [ ]:
# Test with a new suspicious transaction
new_transaction = pd.DataFrame([{
    'transaction_amount': 4500,   # High amount
    'time_of_day': 2,              # 2 AM
    'distance_from_home': 120,     # Far from home
    'num_transactions_today': 12,  # Many transactions
    'is_foreign_transaction': 1,   # Foreign
    'is_online': 1,                # Online
    'credit_score': 550,           # Low credit score
    'age': 25
}])

# Scale and predict
new_scaled = scaler.transform(new_transaction)
best_model_obj = results[best_model]['model']
fraud_prob = best_model_obj.predict_proba(new_scaled)[0][1]
prediction = best_model_obj.predict(new_scaled)[0]

print('=== 🔍 New Transaction Analysis ===')
print(f'Amount: ₹4,500 | Time: 2 AM | Foreign: Yes | Online: Yes')
print(f'\n🤖 Model Prediction: {"🚨 FRAUD DETECTED!" if prediction == 1 else "✅ Legitimate"}')
print(f'📊 Fraud Probability: {fraud_prob*100:.1f}%')

## ✅ Project Complete!

### Key Findings:
- Built and compared **3 ML models** for fraud detection
- Handled **class imbalance** using SMOTE technique
- Used **Precision, Recall, F1, AUC-ROC** for evaluation
- XGBoost/Random Forest outperforms Logistic Regression
- **Recall is prioritized** over Precision in fraud detection (we don't want to miss fraud cases)

### Technologies Used:
- Python, Pandas, NumPy
- Scikit-learn, XGBoost
- Matplotlib, Seaborn
- SMOTE (imbalanced-learn)